# Individual 5-Category Benchmark — Final Winning Architecture (Zero Pooling)
**Project:** 22AI304 / 22AIE304 End-Semester Capstone — Team 11  
**Protocol:** Trains **5 completely separate models** (1 per MVTec LOCO category, strictly single-category `train/good` → single-category `validation/good` calibration → single-category `test/good` vs. `test/logical_anomalies`, **zero pooling**) on the single overall final winning architecture from `05_colab_final_integration.ipynb`.

**Direct Comparison Against Prior Published Individual-Category Baselines (Assignment 2 & 3):**
- A2 **PUAD**: `91.58%` (BB `91.47%`, JB `99.88%`, PP `96.82%`, SB `74.49%`, SC `95.22%`)
- A3 **SALAD**: `91.52%` | **LogiCo**: `87.86%` | **CSAD**: `86.54%` | **GCAD**: `86.52%` | **ComAD**: `76.03%`

In [ ]:
#@title 1. Verify Colab GPU, RAM, Disk & Install Dependencies
!pip install -q timm thop pyyaml scikit-learn pandas
import torch, psutil, shutil, sys
print('=== COLAB SESSION AUDIT (INDIVIDUAL 5-CAT BENCHMARK) ===')
print(f'Python        : {sys.version.split()[0]}')
print(f'PyTorch       : {torch.__version__} | CUDA: {torch.version.cuda}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    native_bf16 = p.major >= 8
    print(f'GPU Model     : {p.name} (CC {p.major}.{p.minor}) | VRAM: {p.total_memory/1024**3:.2f} GiB | Native BF16: {native_bf16}')
else:
    raise RuntimeError('No GPU attached! Runtime -> Change runtime type -> L4 / G4 / A100 GPU.')
ram = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print(f'System RAM    : {ram.available/1e9:.1f} GB free / {ram.total/1e9:.1f} GB total')
print(f'Local SSD     : {disk.free/1e9:.1f} GB free / {disk.total/1e9:.1f} GB total')

In [ ]:
#@title 2. Mount Drive, Pull Latest Repo, Unpack Data & Run Section 3 Audit
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = 'https://github.com/Chimera418/dl-endsem-logical-ad.git'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Endsem_Team11'
DRIVE_RUNS = f'{DRIVE_ROOT}/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)

if not os.path.exists('/content/Endsem/.git'):
    !git clone {REPO_URL} /content/Endsem
else:
    !git -C /content/Endsem pull --ff-only
%cd /content/Endsem

if not os.path.exists('/content/data/mvtec_loco_anomaly_detection/breakfast_box'):
    if os.path.exists('/content/data/breakfast_box'):
        os.environ['ENDSEM_LOCO_ROOT'] = '/content/data'
    else:
        os.makedirs('/content/data/mvtec_loco_anomaly_detection', exist_ok=True)
        !tar -xf "{DRIVE_ROOT}/data/mvtec_loco_anomaly_detection.tar.xz" -C /content/data/mvtec_loco_anomaly_detection/
        os.environ['ENDSEM_LOCO_ROOT'] = '/content/data/mvtec_loco_anomaly_detection'
else:
    os.environ['ENDSEM_LOCO_ROOT'] = '/content/data/mvtec_loco_anomaly_detection'

if not os.path.exists('/content/data/imagenet_penalty'):
    os.makedirs('/content/data/imagenet_penalty', exist_ok=True)
    !unzip -q "{DRIVE_ROOT}/data/imagenet_penalty_10k.zip" -d /content/data/imagenet_penalty

os.environ['ENDSEM_IMAGENET_ROOT'] = '/content/data/imagenet_penalty'
os.environ['ENDSEM_TEACHER_PATH'] = f'{DRIVE_ROOT}/weights/teacher_pdn_s.pt'

!python scripts/verify_env.py

In [ ]:
#@title 3. Train & Evaluate 5 Separate Per-Category Models on the Final Winner (Zero Pooling)
# Update WINNER_CONFIG after E20/E21 results are analyzed!
WINNER_CONFIG = 'configs/E21_fpn_crisscross_transformer.yaml'  # <-- set to final winner config

import subprocess, json, pathlib, yaml
CATEGORIES = ['breakfast_box', 'juice_bottle', 'pushpins', 'screw_bag', 'splicing_connectors']
exp_id = yaml.safe_load(open(WINNER_CONFIG))['exp_id']
scores = {}

for cat in CATEGORIES:
    subprocess.run(
        ['python', 'scripts/run_experiment.py', '--config', WINNER_CONFIG, '--category', cat, '--mode', 'train', '--drive-sync-dir', DRIVE_RUNS],
        check=True,
    )
    res_path = pathlib.Path('experiments') / f'{exp_id}_ind_{cat}' / 'test_results.json'
    d = json.loads(res_path.read_text())
    scores[cat] = d['metrics']['per_category_auroc'][cat]

mean_ind = sum(scores.values()) / len(scores)
print('\n' + '=' * 66)
print(f'INDIVIDUAL 5-CATEGORY BENCHMARK (ZERO POOLING): {exp_id}')
print('=' * 66)
for cat, v in scores.items():
    print(f'  {cat:22s} : {v * 100:6.2f}%')
print('-' * 66)
print(f'  {"Individual Mean AUROC":22s} : {mean_ind * 100:6.2f}%')
print(f'  {"Delta vs A2 PUAD (91.58%)":22s} : {(mean_ind - 0.9158) * 100:+6.2f} pp')
print(f'  {"Delta vs A3 SALAD (91.52%)":22s} : {(mean_ind - 0.9152) * 100:+6.2f} pp')
print('=' * 66)